# 1 · Explore the data

Scratch space for **Milestone 1**. Nothing here is graded - the graded code is `services/dataset_service.py`.

Finish `load_dataset()` and `profile_dataset()` first, then use this notebook to *look* at what you loaded. The aim is to leave with three observations about the data that will matter later.

In [ ]:
# Run this cell first, every time. It lets the notebook import the assignment's
# code, and re-imports your .py files automatically whenever you save them.
%load_ext autoreload
%autoreload 2

import os, sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "agent").is_dir() else Path.cwd().parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("working in", ROOT)

## Load and profile

Safe to run as often as you like: the load starts from a clean slate every time.

In [ ]:
from services.dataset_service import load_dataset, profile_dataset

print(load_dataset())
profile_dataset()

## Look at the tables

`connect()` gives you an ordinary read-write connection to **your own** database. pandas turns a query into a DataFrame.

In [ ]:
import pandas as pd
from services.dataset_service import connect

def sql(query: str) -> pd.DataFrame:
    with connect() as conn:
        cursor = conn.execute(query)
        return pd.DataFrame(cursor.fetchall(), columns=[c.name for c in cursor.description])

sql("SELECT * FROM student_performance LIMIT 8")

## Observation 1 - not every paper is out of 100

What does that mean for an average of `marks`?

In [ ]:
sql("""
    SELECT sd.subject_type, sp.total_marks, COUNT(*) AS records,
           ROUND(AVG(sp.marks), 2)                          AS average_raw_marks,
           ROUND(AVG(sp.marks * 100.0 / sp.total_marks), 2) AS average_percentage
    FROM student_performance sp JOIN subject_details sd ON sd.subject = sp.subject
    GROUP BY sd.subject_type, sp.total_marks
""")

## Observation 2 - some values are missing

Which columns? How many? What might an empty `marks` mean - and an empty `attendance`? Write your guess down. In Milestone 4 you will find out whether you were right.

In [ ]:
sql("""
    SELECT COUNT(*) FILTER (WHERE marks IS NULL)      AS marks_missing,
           COUNT(*) FILTER (WHERE attendance IS NULL) AS attendance_missing
    FROM student_performance
""")

## Observation 3 - the same number, three ways

Below are three reasonable-looking ways to compute a department's *pass percentage*. They disagree. Only one matches the college's rules - and nothing in the data tells you which.

In [ ]:
sql("""
    SELECT sp.department,
           ROUND(100.0 * COUNT(*) FILTER (WHERE sp.marks >= 40) / COUNT(*), 2)                          AS raw_marks_40,
           ROUND(100.0 * COUNT(*) FILTER (WHERE sp.marks * 100.0 / sp.total_marks >= 40) / COUNT(*), 2) AS percent_40,
           ROUND(100.0 * COUNT(*) FILTER (WHERE sp.marks * 100.0 / sp.total_marks >= 40)
                 / COUNT(sp.marks), 2)                                                               AS percent_40_absent_ignored
    FROM student_performance sp
    GROUP BY sp.department ORDER BY sp.department
""")

Hold on to that table. The agent you are building will be asked this exact question, and the difference between these columns is the difference between an agent that guesses and one that looks the rule up.